# Introduction to MCP

- The MCP framework connects agents to external capabilities using a client-server protocol.
- Through this protocol, MCP servers can offer,
      tools (which can modify state or connect to external systems), prompts (reusable templates that guide tool usage), and
      resources (like data and content).
- On the client side, agents use LLMs to plan which tool to invoke and format the necessary parameters for the request.
- MCP servers securely manage these requests, acting as an interface to underlying systems (such as databases) and replying to the client in a standardized format.

In [1]:
%pip install -q  mcp==1.15.0 \
                 langchain==0.3.20 \
                 langchain-mcp-adapters==0.1.11 \
                 langgraph==0.6.9 \
                 langchain_openai==0.3.35 \
                 langgraph==0.6.9 \
                 langchain-community==0.3.19




Reason for being yanked: <none given>
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.3 requires langchain-core<2.0.0,>=1.2.19, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.3 requires langchain-text-splitters<2.0.0,>=1.1.1, but you have langchain-text-splitters 0.3.11 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## Learning Objective

Implement an MCP-powered Agentic System for generating personalized marketing customer pitches, showcasing the integration of external storage, server-hosted tools with a single agent client.

## Business Case: Marketing Pitch AI Agent
In the marketing industry, crafting personalized customer pitches is essential for engagement and conversion. Teams often spend hours researching customer profiles, tailoring messages, and iterating based on feedback. Traditional methods, rely on manual templates and static tools.

## Problem Scenario

Fragmented research and lack of personalization may result in low response rates, and missed opportunities.  Pitches may overlook key customer details like preferences or recent interactions, causing misalignment with brand voice or campaign goals. This leads to higher churn, wasted ad spend, and suboptimal ROI in competitive markets.

## Proposed Solution

 An automated Marketing Customer Pitch AI Agent powered by an MCP (Model Context Protocol) Agentic System. This system gathers user inputs from external storage (e.g. - User 360°), leverages MCP server-hosted tools for computations (e.g., customer research, pitch scoring), and enables dynamic refinement to generate and adjust tailored pitches.Likely benefits include faster pitch creation, higher personalization accuracy, improved conversion rates, and scalable marketing operations.

In [2]:
import os
import asyncio
import threading
import time


from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

from mcp.server.fastmcp import FastMCP
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent

from langchain_core.tools import tool
from langchain_mcp_adapters.tools import to_fastmcp

In [3]:
openai_api_key = "gl-U2FsdGVkX180NaoooXHzbLgUi1B+6tqdtZU/pI4iZOY+bHLZjlMgPMFOqz6VY0RQ"

### Implementation Plan

The notebook implements an agentic workflow for marketing pitch generation using the langgraph package and MCP for server-hosted tool integration. The execution workflow consists of the following steps:

1. **MCP Server Setup**: Define and expose tools for customer research, pitch scoring, and refinement on the server.
2. **Client Integration**: Load MCP tools into a LangGraph ReAct agent for conversational interaction.
3. **Demonstration**: Run the session to showcase pitch creation & refinment.


## External Database (Simulation)


In [4]:
# Simulated customer database for research_customer tool
CUSTOMER_DB = {
    "Acme Corp": "Leading manufacturer of road runner catching devices. Known for innovation and quality.",
    "Beta LLC": "Specializes in eco-friendly home products, strong community presence.",
    "Gamma Tech": "Emerging startup in AI-driven logistics software. Focuses on automation and scalability for e-commerce.",
    "Delta Foods": "Family-owned organic food producer emphasizing sustainable farming and direct-to-consumer sales.",
    "Epsilon Media": "Digital marketing agency with expertise in social media campaigns and content creation for B2B clients.",
    "Zeta Renewables": "Provider of solar energy solutions for residential and commercial buildings. Committed to green energy transition."
}

## Server Setup



### Define Tools

Tools for customer info,pitch and scoring are defined here.
_Example Tool Definitions:_
- `research_customer`: This tool will mock data retrieval of customer preferences and history.
- `initial_pitch_prompt` : This tool generates an initial sales pitch prompt based on customer.
- `score_pitch`: Evaluates the effectiveness of a pitch based on predefined criteria.
- `refine_pitch`: Adjusts the pitch based on user feedback to improve its effectiveness and alignment with customer needs.

### Tool: Research Customer
Tool to fetch/mock researched customer data ( from customer 360-degree data storage)


In [5]:
# Define LangGraph tools using @tool decorators
@tool
def research_customer(name: str) -> str:
    """Lookup customer profile"""
    return CUSTOMER_DB.get(name, "No customer data found.")

### Tool: Score Pitch

Score the genrated pitch on personalization, engagement, and length.



In [6]:
@tool
def score_pitch(pitch: str) -> str:
    """Use LLM to score pitch"""
    prompt = (
        f"Evaluate the following sales pitch on persuasiveness, clarity, and relevance. "
        f"Score each 1 to 10 and output JSON only:\n{pitch}\n\n"
        "{\"persuasiveness\": int, \"clarity\": int, \"relevance\": int}"
    )
    messages = [HumanMessage(content=prompt)]
    response = llm.invoke(messages)  # Use .invoke() for sync call
    import re
    json_match = re.search(r"\{.*\}", response.content, re.DOTALL)
    if json_match:
        return json_match.group(0)
    else:
        # Fallback with error handling
        return '{"persuasiveness": 5, "clarity": 5, "relevance": 5}'

### Tool : Refine Pitch

Refine the pitch based on user feedback. Mock refinement for demo


In [7]:
@tool
def refine_pitch(pitch: str, feedback: str) -> str:
    """Use LLM to rewrite pitch with feedback"""

    prompt = (
        f"You are an expert sales copywriter specializing in personalized B2B pitches. "
        f"Rewrite the following sales pitch to explicitly incorporate the feedback: make targeted changes "
        f"such as shortening sentences, adding innovative elements, or enhancing calls-to-action. "
        f"Ensure the revised pitch is engaging, concise (under 150 words), and structured as an email "
        f"with a compelling subject line, greeting, body (2-4 paragraphs), and professional sign-off. "
        f"Focus on value, personalization, and the customer's needs to boost persuasiveness.\n\n"
        f"Original Pitch:\n{pitch}\n\n"
        f"Feedback to Incorporate: {feedback}\n\n"
        f"Revised Pitch:"
    )
    messages = [HumanMessage(content=prompt)]
    response = llm.invoke(messages)  # Use .invoke() for sync call
    return response.content

### Tool : Initial Pitch

In [8]:
@tool
def initial_pitch_prompt(customer_name: str, customer_info: str) -> str:
    """Generate an initial sales pitch prompt based on customer name and information."""
    return (
        f"Create a persuasive, personalized sales pitch for {customer_name}. "
        f"Incorporate this customer info: {customer_info}. "
        f"Keep it concise, engaging, and focused on value."
    )

In [9]:
# Convert LangChain tools to FastMCP
langgraph_tools = [research_customer, score_pitch, refine_pitch, initial_pitch_prompt]
fastmcp_tools = [to_fastmcp(t) for t in langgraph_tools]

In [10]:
# Create MCP instance for server
mcp = FastMCP("MCP Marketing Pitch Tools", host="127.0.0.1", port=8000, tools=fastmcp_tools)

### Start Server

In [12]:
# Start the MCP server in a background thread
thread = threading.Thread(
    target=mcp.run,
    kwargs={
        "transport": "streamable-http"
    },
    daemon=True
)
thread.start()

# Brief pause to allow the server to fully start
time.sleep(2)
print("MCP server started on http://127.0.0.1:8000/mcp")

INFO:     Started server process [36834]
INFO:     Waiting for application startup.
ERROR:    Traceback (most recent call last):
  File "/opt/homebrew/lib/python3.11/site-packages/starlette/routing.py", line 638, in lifespan
    async with self.lifespan_context(app) as maybe_state:
  File "/opt/homebrew/Cellar/python@3.11/3.11.15_1/Frameworks/Python.framework/Versions/3.11/lib/python3.11/contextlib.py", line 210, in __aenter__
    return await anext(self.gen)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/lib/python3.11/site-packages/mcp/server/streamable_http_manager.py", line 101, in run
    raise RuntimeError(
RuntimeError: StreamableHTTPSessionManager .run() can only be called once per instance. Create a new instance if you need to run again.

ERROR:    Application startup failed. Exiting.


MCP server started on http://127.0.0.1:8000/mcp


## Client Setup  

### LLM

In [13]:
llm = ChatOpenAI(
    api_key=openai_api_key,
    base_url="https://aibe.mygreatlearning.com/openai/v1",
    model='gpt-4o-mini',
    temperature=0
)

###Load Tools and Create Agent

In [14]:
async def run_pitch_demo(customer_name):
    # Keep session open for entire demo by wrapping everything in async with
    async with streamablehttp_client("http://127.0.0.1:8000/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            mcp_tools = await load_mcp_tools(session)
            agent = create_react_agent(llm, tools=mcp_tools)


            # Step 1: Use agent to research and generate initial pitch (end-to-end, non-streaming)
            generate_msg = HumanMessage(content=f"Generate a personalized sales pitch for {customer_name}. Research the customer first.")
            try:
                result = await agent.ainvoke({"messages": [generate_msg]})
                # Extract the last AIMessage content from the final state
                messages = result["messages"]
                pitch_msg = next((m for m in reversed(messages) if hasattr(m, 'content') and m.type == 'ai'), None)
                if pitch_msg:
                    pitch = pitch_msg.content
                    print(f"Initial Pitch:\n{pitch.strip()}")
                else:
                    print("No AI response found in agent output.")
                    return  # Early exit if no pitch
            except Exception as e:
                print(f"Error in agent invocation: {e}")
                return

            # Step 2: Score the pitch directly via tool (find by name for robustness, use ainvoke)
            try:
                score_tool = next(t for t in mcp_tools if t.name == "score_pitch")
                score = await score_tool.ainvoke({"pitch": pitch})
                print(f"Pitch score:\n{score}")
            except Exception as e:
                print(f"Error scoring pitch: {e}")

            # Step 3: Refine via tool
            feedback = f"Redraft pitch to improve {score}."
            try:
                refine_tool = next(t for t in mcp_tools if t.name == "refine_pitch")
                refined_pitch = await refine_tool.ainvoke({"pitch": pitch, "feedback": feedback})
                print(f"Refined Pitch:\n{refined_pitch}")
            except Exception as e:
                print(f"Error refining pitch: {e}")
                refined_pitch = None  # Set to None on failure

            # Step 4: Re-score (only if refinement succeeded)
            if refined_pitch:
                try:
                    refined_score = await score_tool.ainvoke({"pitch": refined_pitch})
                    print(f"Refined Pitch score:\n{refined_score}")
                except Exception as e:
                    print(f"Error re-scoring pitch: {e}")
            else:
                print("Skipping re-score due to refinement failure.")



### Execution

In [16]:
# Execute the demo
# By passing valid customer-name
await run_pitch_demo("Beta LLC")

[05/08/26 16:36:06] INFO     Created new transport with session ID:                  ]8;id=10835560;file:///opt/homebrew/lib/python3.11/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=10835561;file:///opt/homebrew/lib/python3.11/site-packages/mcp/server/streamable_http_manager.py#233\233]8;;\
                             7cb8845ed46a4473ba7957f8af9fb799                                                      

INFO:     127.0.0.1:55896 - "POST /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"         ]8;id=10835566;file:///opt/homebrew/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10835567;file:///opt/homebrew/lib/python3.11/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Received session ID: 7cb8845ed46a4473ba7957f8af9fb799           ]8;id=10835572;file:///opt/homebrew/lib/python3.11/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=10835573;file:///opt/homebrew/lib/python3.11/site-packages/mcp/client/streamable_http.py#134\134]8;;\

                    INFO     Negotiated protocol version: 2025-06-18                         ]8;id=10835578;file:///opt/homebrew/lib/python3.11/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=10835579;file:///opt/homebrew/lib/python3.11/site-packages/mcp/client/streamable_http.py#146\146]8;;\

...

                    INFO     Processing request of type ListToolsRequest                              ]8;id=10835728;file:///opt/homebrew/lib/python3.11/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=10835729;file:///opt/homebrew/lib/python3.11/site-packages/mcp/server/lowlevel/server.py#664\664]8;;\

INFO:     127.0.0.1:58856 - "POST /mcp HTTP/1.1" 200 OK


                    INFO     Processing request of type ListToolsRequest                              ]8;id=10835734;file:///opt/homebrew/lib/python3.11/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=10835735;file:///opt/homebrew/lib/python3.11/site-packages/mcp/server/lowlevel/server.py#664\664]8;;\

[05/08/26 16:51:07] INFO     Terminating session: 278b58ba134143c4ba04dd9a88f7f4d3           ]8;id=10835740;file:///opt/homebrew/lib/python3.11/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=10835741;file:///opt/homebrew/lib/python3.11/site-packages/mcp/server/streamable_http.py#630\630]8;;\

INFO:     127.0.0.1:58975 - "DELETE /mcp HTTP/1.1" 200 OK
